<h1 style="color: #CEDDF4;">DEPI Round 4, MS Data Engineer and AI Track</h1>
<h2 style="color: #CEDDF4;" >Final Project: Gold and Oil Prediction System</h2>
<h3 style="color: #CEDDF4;" >Python Code</h3>

<h4 style="color: #CEDDF4;" >1. Import Libraries</h4>

In [13]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import requests
import pandas as pd
import time
import csv
from itertools import zip_longest
import math
from pathlib import Path
import os
import yfinance as yf
from datetime import datetime

In [2]:
base_dir = Path.cwd()
#raw data path to contain all raw files gathered either manually, web scraping or APIs
raw_data_dir=base_dir/'raw data'
#cleaned data path, will be separated into two main folders which are: market data (contains the data with daily logs) and macroeconomic data
##cleaned market data
cleaned_market_data_dir=base_dir/'cleaned data'/'market_data'
##cleaned macroeconomic data
cleaned_macro_data_dir=base_dir/'cleaned data'/'macroeconomic_data'

In [14]:
#today date 
today = datetime.now().strftime("%d/%m/%Y")

<h4 style="color: #CEDDF4;" >2. Egyptian Market Data</h4>
<h5 style="color: #CEDDF4;" >2.1 CPI Rate from IMF</h5>

In [ ]:
#this is the API code for getting the CPI (Consumer Products Index) for Egypt from IMF (International Monetary Fund)
#this API is free

#function to be used at any country with any of published indices of IMF

def imf_data(country,index):
    url=f'https://www.imf.org/external/datamapper/api/v1/{index}/{country}'
    response = requests.get(url)
    data=response.json()
    output=data['values'][index][country]
    df = pd.DataFrame(list(output.items()), columns=['year', index])
    return df

#function to be used for IMF data cleaning 
#cl --> cleaned

def cl_imf_data(dataframe,index,country,start_period,end_period):
    dataframe=dataframe[(dataframe['year']>=start_period)&(dataframe['year']<=end_period)]
    dataframe['year']=pd.to_datetime(dataframe['year'])
    dataframe=dataframe.rename(columns={dataframe.columns[1]:'value'})
    dataframe.insert(loc=0, column='id', value=range(1, len(dataframe) + 1)) #to create the key for the values
    dataframe.insert(loc=1,column='region',value=country)
    dataframe.insert(loc=2,column='ticker',value=index)
    return dataframe

In [5]:
#get CPI index data for egypt
egy_cpi_df = imf_data('EGY','PCPIPCH')
egy_cpi_df.to_csv(raw_data_dir/"egy_cpi.csv", index=False)

#cleaning of CPI index data for egypt
cl_egy_cpi_df=cl_imf_data(egy_cpi_df,'cpi_imf','egypt','2015','2025')
cl_egy_cpi_df.to_csv(cleaned_macro_data_dir/"egy_cpi.csv", index=False)

<h5 style="color: #CEDDF4;" >2.2 Interest Rate from CBE</h5>

In [45]:
#get egyptian data from cbe (central bank of egypt)

#the following options are countermeasure for detecting that we're using code to navigate the website
options=Options()
#options.add_argument("--headless") #To make the code operates in the background
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches",["enable-automation"])
options.add_experimental_option("useAutomationExtension",False)
#option to save the downloaded files into data folders 
prefs= {
    "download.default_directory": str(raw_data_dir),
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
}
options.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=options)

#function to add the needed data for cbe statistics
## will not work on exchange rates till now
def cbe_data(target_data,file_name,start_date):
    cbe_url=f'https://www.cbe.org.eg/en/economic-research/statistics/{target_data.replace(" ","-")}/historical-data'
    driver.get(cbe_url)
    #start_date = "01/01/2016"
    driver.execute_script(f"document.getElementById('fromDate').value = '{start_date}';")
    #end_date="31/12/2025"
    driver.execute_script(f"document.getElementById('toDate').value = '{today}';")
    driver.find_element(By.XPATH,'/html/body/div[1]/section[3]/div[1]/form/div[1]/div/button/div/span[2]').click()
    time.sleep(1)
    driver.find_element(By.XPATH,'//*[@id="historicalDataForm"]/div[2]/button[1]/span').click()
    downloaded_filename = max(raw_data_dir.glob('*'), key=os.path.getmtime)
    downloaded_filename.rename(raw_data_dir / f"{file_name}.xlsx")
    return file_name

In [46]:
cbe_inflation=cbe_data('inflation rates','egy_cbe_inflation_rate','01/01/2016')

In [ ]:
def cl_cbe_data(df,country):
    df = pd.read_excel(raw_data_dir/'egy_cbe_inflation_rate.xlsx')
    df=df.drop(columns=['Unnamed: 3','Unnamed: 4'])
    df=df.rename(columns={df.columns[0]:'date',df.columns[1]:'headline_inflation_yy',df.columns[2]:'core_inflation_yy'})
    df['date'] = pd.to_datetime(df['date'], format='%b %Y', errors='coerce').dt.strftime('%d/%m/%Y') #this was made to convert the raw date (feb. 2026 to 01/02/2026)
    df = df.dropna(subset=['date'])
    df = df.reset_index(drop=True)
    #to transpose the values of the multiple tickers
    #column_values=[col for col in df.columns if col != 'date']
    #df=df.melt(id_vars=['date'],value_vars=column_values,var_name='ticker',value_name='value')
    #df['value'] = df['value'].str.replace('%', '').astype(float)
    df.insert(loc=1,column='region',value=country)
    df = df.sort_values(by='date', key=lambda x: pd.to_datetime(x, format='%d/%m/%Y')).reset_index(drop=True) #this was made to sort the dates from oldest to newest
    df.insert(loc=0, column='id', value=range(1, len(df) + 1))
    return df

In [56]:
cl_cbe_data_df=cl_cbe_data(cbe_inflation,'egypt')
cl_cbe_data_df.to_csv(cleaned_macro_data_dir/"egy_cbe_inflation_rates.csv", index=False)

<h5 style="color: #CEDDF4;" >2.3 USD/EGP Exchange Rate from CBE</h5>

In [57]:
#the following options are countermeasure for detecting that we're using code to navigate the website
options=Options()
#options.add_argument("--headless") #To make the code operates in the background
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches",["enable-automation"])
options.add_experimental_option("useAutomationExtension",False)
#option to save the downloaded files into data folders 
prefs= {
    "download.default_directory": str(raw_data_dir),
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
}
options.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=options)

#get exchange rate between usd and egp from cbe
def cbe_usdegp_er(start_date,file_name):
    #file_name='egy_usdegp_rate'
    er_cbe_url='https://www.cbe.org.eg/en/markets/foreign-exchange/foreign-exchange-historical-data'
    driver.get(er_cbe_url)
    time.sleep(5)
    #start_date = "01/01/2016"
    driver.execute_script(f"document.getElementById('fromDate').value = '{start_date}';")
    #end_date="31/12/2025"
    driver.execute_script(f"document.getElementById('toDate').value = '{today}';")
    driver.find_element(By.XPATH,'/html/body/div[1]/section[3]/div[1]/form/div[1]/div/button/div/span[2]/span').click()
    time.sleep(1)
    driver.find_element(By.XPATH,'//*[@id="historicalDataForm"]/div[2]/button[1]').click()
    time.sleep(2)
    downloaded_filename = max(raw_data_dir.glob('*'), key=os.path.getmtime)
    time.sleep(2)
    downloaded_filename.rename(raw_data_dir /f"{file_name}{downloaded_filename.suffix}")
    return file_name


In [59]:
cbe_usdegp_er('01/01/2016','egy_cbe_egpusd_rate')

'egy_cbe_egpusd_rate'

In [ ]:
def cl_cbe_usdegp(df, country, ticker):
    df = pd.read_excel(raw_data_dir / 'egy_cbe_egpusd_rate.xlsx')
    df = df.rename(columns={df.columns[0]: 'date', df.columns[1]: 'value'})
    df.insert(loc=0, column='region', value=country)
    df.insert(loc=1, column='ticker', value=ticker)
    df['date'] = pd.to_datetime(df['date'], format='mixed', dayfirst=True, errors='coerce') #convert the date in the excel file from cbe website
    df = df.dropna(subset=['date'])
    df = df.sort_values('date').reset_index(drop=True)
    df = df.set_index('date')
    df = df[~df.index.duplicated(keep='first')]  # remove duplicate dates based on some errors appeared in the run trials
    full_dates = pd.date_range(start='2016-01-01', end=pd.Timestamp.today(), freq='D')
    df = df.reindex(full_dates)
    df = df.bfill()
    df = df.reset_index().rename(columns={'index': 'date'})
    df['date'] = df['date'].dt.strftime('%d/%m/%Y')
    df.insert(loc=0, column='id', value=range(1, len(df) + 1))
    return df

In [69]:
cl_cbe_usdegp_rate = cl_cbe_usdegp('usdegp_rate','egypt','egpusd')
cl_cbe_usdegp_rate.to_csv(cleaned_market_data_dir/"egy_egpusd_rate.csv", index=False)

<h5 style="color: #CEDDF4;" >2.4 Subsidies on Electrical / Petroleum Products in Egyptian Budget from MOF</h5>

In [ ]:
#this data was extracted manually from egyptian budget which is published on ministry of finance (mof) website
def cl_egy_budget(filename1,filename2,cleaned_filename):
    df1 = pd.read_excel(raw_data_dir / f'{filename1}.xlsx')
    keyword1 = filename1.split('_on_')[1].split('_expected')[0]
    df1 = df1.rename(columns={df1.columns[0]: 'date', df1.columns[1]: f"planned_subsidy_{keyword1}",df1.columns[2]: f'actual_subsidy_{keyword1}'})
    df1[f'actual_subsidy_{keyword1}'] = df1[f'actual_subsidy_{keyword1}'].fillna(df1[f'planned_subsidy_{keyword1}'])
    df2 = pd.read_excel(raw_data_dir / f'{filename2}.xlsx')
    keyword2 = filename2.split('_on_')[1].split('_expected')[0]
    df2 = df2.rename(columns={df2.columns[0]: 'date', df2.columns[1]: f"planned_subsidy_{keyword2}",df2.columns[2]: f'actual_subsidy_{keyword2}'})
    df2[f'actual_subsidy_{keyword2}'] = df2[f'actual_subsidy_{keyword2}'].fillna(df2[f'planned_subsidy_{keyword2}'])
    df=pd.merge(df1,df2,on='date',how='inner')
    #to transpose the values of the multiple tickers
    #column_values=[col for col in df.columns if col != 'date']
    #df=df.melt(id_vars=['date'],value_vars=column_values,var_name='ticker',value_name='value_in_millions')
    df.insert(loc=0,column='region',value='egypt')
    df = df.sort_values('date').reset_index(drop=True)
    df.insert(loc=0, column='id', value=range(1, len(df) + 1))
    df.to_csv(cleaned_macro_data_dir/f"{cleaned_filename}.csv", index=False)
    return

In [75]:
cl_egy_budget('egy_subsidies_on_electricity_expected_vs_actual','egy_subsidies_on_petroleum_products_expected_vs_actual','egy_subsidies_in_budget')

<h4 style="color: #CEDDF4;" >3. Yahoo Finance Data</h4>

In [107]:
#retrieve the global stock and commodity data from yfinance
def yf_data(region,table_ticker,target_data,y_ticker,startdate):
    today_yf = pd.Timestamp.today().strftime('%Y-%m-%d') #today in yahoo finance is different from global today variable because of sorting YMD
    df=yf.download(f"{y_ticker}",start=f'{startdate}',end=today_yf)

    if isinstance(df.columns,pd.MultiIndex):
        df.columns = df.columns.droplevel(1)

    df = df.reset_index()
    df.to_csv(raw_data_dir/f"global_{target_data}_prices.csv", index=False)
    #cleaning of retrieved data from yfinance
    cl_df=df.copy()
    cl_df=cl_df.drop('Volume',axis=1,errors='ignore')
    #cl_df = cl_df.drop(index=0)  # drop second row (index 0 after read_csv)
    cl_df = cl_df.reset_index(drop=True)
    cl_df =cl_df.rename(columns={'Date':'date','Close':'price_usd','High':'high_usd','Low':'low_usd','Open':'open_usd'})
    cl_df['date'] = pd.to_datetime(cl_df['date'])
    cl_df = cl_df.set_index('date')
    full_dates = pd.date_range(start=f'{startdate}',end=today_yf, freq='D')# create full date range
    cl_df.index = pd.to_datetime(cl_df.index)
    cl_df = cl_df.reindex(full_dates)
    cl_df = cl_df.bfill().ffill()
    cl_df = cl_df.reset_index().rename(columns={'index':'date'})
    #to transpose the values of the multiple tickers
    #column_values = [col for col in cl_df.columns if col != 'date']
    #cl_df = cl_df.melt(id_vars=['date'], value_vars=column_values, var_name='ticker', value_name='value_in_usd')
    cl_df.insert(loc=0, column='ticker', value=f'{table_ticker}')
    cl_df.insert(loc=0, column='region', value=f'{region}')
    cl_df = cl_df.sort_values('date').reset_index(drop=True)
    cl_df.insert(loc=0, column='id', value=range(1, len(cl_df) + 1))
    cl_df.to_csv(cleaned_market_data_dir/f"global_{target_data}_prices.csv", index=False)
    return 

<h4 style="color: #CEDDF4;" >4 Gold Data</h5>
<h5 style="color: #CEDDF4;" >4.1 Global Gold Data</h5>

In [108]:
#yahoo finance ticker is GC=F
yf_data('global','gold','gold','GC=F','2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >4.2 Egyptian Gold Data</h5>

In [94]:
#this data was downloaded manually from https://www.investing.com/currencies/xau-egp-historical-data
egy_gold_df=pd.read_csv(raw_data_dir/ 'egy_gold_prices.csv')
egy_gold_df=egy_gold_df.drop('Vol.',axis=1)
egy_gold_df=egy_gold_df.drop('Change %',axis=1)
egy_gold_df = egy_gold_df.drop(index=0)
egy_gold_df = egy_gold_df.reset_index(drop=True)
egy_gold_df =egy_gold_df.rename(columns={'Date':'date','Price':'price_oz_egp','High':'high_price_egp','Low':'low_price_egp','Open':'open_price_egp'})
egy_gold_df['date'] = pd.to_datetime(egy_gold_df['date'], format='mixed', dayfirst=False)
egy_gold_df = egy_gold_df.set_index('date')
full_dates = pd.date_range(start='01-01-2016',end=today, freq='D')# create full date range
egy_gold_df.index = pd.to_datetime(egy_gold_df.index) 
egy_gold_df = egy_gold_df.reindex(full_dates)
egy_gold_df = egy_gold_df.bfill().ffill()
egy_gold_df = egy_gold_df.reset_index().rename(columns={'index':'date'})  
#to transpose the values of the multiple tickers
#column_values = [col for col in egy_gold_df.columns if col != 'date']
#egy_gold_df = egy_gold_df.melt(id_vars=['date'], value_vars=column_values, var_name='ticker', value_name='value_in_egp')
egy_gold_df.insert(loc=0, column='ticker', value='gold')
egy_gold_df.insert(loc=0, column='region', value='egypt')
egy_gold_df = egy_gold_df.sort_values('date').reset_index(drop=True)
egy_gold_df.insert(loc=0, column='id', value=range(1, len(egy_gold_df) + 1))
egy_gold_df.to_csv(cleaned_market_data_dir/"egy_gold_prices.csv", index=False)

In [95]:
egy_gold_df

,id,region,ticker,date,price_oz_egp,open_price_egp,high_price_egp,low_price_egp
0,1,egypt,gold,2016-01-01,"8,301.68","8,301.68","8,301.68","8,301.68"
1,2,egypt,gold,2016-01-02,"8,407.85","8,341.71","8,485.23","8,314.65"
2,3,egypt,gold,2016-01-03,"8,407.85","8,341.71","8,485.23","8,314.65"
3,4,egypt,gold,2016-01-04,"8,407.85","8,341.71","8,485.23","8,314.65"
4,5,egypt,gold,2016-01-05,"8,432.60","8,408.26","8,475.37","8,398.91"
...,...,...,...,...,...,...,...,...
3722,3723,egypt,gold,2026-03-11,"268,497.63","269,792.75","271,576.06","267,127.69"
3723,3724,egypt,gold,2026-03-12,"265,853.25","268,508.44","272,539.75","264,561.72"
3724,3725,egypt,gold,2026-03-13,"265,853.25","268,508.44","272,539.75","264,561.72"
3725,3726,egypt,gold,2026-03-14,"265,853.25","268,508.44","272,539.75","264,561.72"


<h4 style="color: #CEDDF4;" >5 Oil Data</h5>
<h5 style="color: #CEDDF4;" >5.1 Brent Oil Data</h5>

In [ ]:
#yahoo finance ticker is BZ=F
yf_data('global','brent','brent_oil','BZ=F','2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >5.2 West Texas Intermediate (WTI) Oil Data</h5>

In [ ]:
#yahoo finance ticker is CL=F
yf_data('global','wti','wti_price','CL=F','2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >5.3 OPEC Basket Oil Data</h5>

In [122]:
#we will get the data using fred api key which is free and available
#api key is ba5f567693ff05a67920dda63bb2affb
def fred_opec(api_key, startdate):
    series_id = 'POILBREUSDM' #this is the series id for opec basket prices in fred
    today_fred = datetime.now().strftime('%Y-%m-%d')  # to change the today from global variable to one that can be used with fred api
    url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&observation_start={startdate}&observation_end={today_fred}&api_key={api_key}&file_type=json"
    response = requests.get(url)
    data = response.json()
    df = pd.DataFrame(data['observations'])
    df = df.rename(columns={'value': 'price_usd'})
    df['price_usd'] = pd.to_numeric(df['price_usd'], errors='coerce')
    df['date'] = pd.to_datetime(df['date']) 
    df = df[['date', 'price_usd']].set_index('date')
    df.to_csv(raw_data_dir/"opec_oil_prices.csv", index=False)
    cl_df = df.copy()
    full_dates = pd.date_range(start=startdate, end=today_fred, freq='D')
    cl_df = cl_df.reindex(full_dates).bfill().ffill()
    cl_df = cl_df.reset_index().rename(columns={'index': 'date'})
    cl_df.insert(0, 'region', 'opec')
    cl_df.insert(1, 'ticker', 'opec_basket')
    cl_df.insert(0, 'id', range(1, len(cl_df) + 1))
    cl_df['date'] = cl_df['date'].dt.strftime('%d/%m/%Y')
    cl_df.to_csv(cleaned_market_data_dir/"opec_oil_prices.csv", index=False)
    return cl_df


In [123]:
fred_opec('ba5f567693ff05a67920dda63bb2affb', '2016-01-01')

,id,region,ticker,date,price_usd
0,1,opec,opec_basket,01/01/2016,32.045238
1,2,opec,opec_basket,02/01/2016,33.762381
2,3,opec,opec_basket,03/01/2016,33.762381
3,4,opec,opec_basket,04/01/2016,33.762381
4,5,opec,opec_basket,05/01/2016,33.762381
...,...,...,...,...,...
3722,3723,opec,opec_basket,11/03/2026,64.594091
3723,3724,opec,opec_basket,12/03/2026,64.594091
3724,3725,opec,opec_basket,13/03/2026,64.594091
3725,3726,opec,opec_basket,14/03/2026,64.594091


<h4 style="color: #CEDDF4;" >6 Exchange Rate Data</h5>
<h5 style="color: #CEDDF4;" >6.1 Chinese Yuan vs USD</h5>

In [125]:
#yahoo finance ticker is CNY=X
yf_data('global','cnyusd','cnyusd_rate','CNY=X', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >6.2 EURO vs USD</h5>

In [130]:
#yahoo finance ticker is EUR=X
yf_data('global','eurusd','eurusd_rate','EUR=X', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >6.4 Japanese Yen vs USD</h5>

In [131]:
#yahoo finance ticker is JPY=X
yf_data('global','jpyusd','jpyusd_rate','JPY=X', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >6.5 Pound Sterling vs USD</h5>

In [132]:
#yahoo finance ticker is GBP=X
yf_data('global','gbpusd','gbpusd_rate','GBP=X', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >6.6 Norwegian Kroner vs USD</h5>

In [133]:
#yahoo finance ticker is NOK=X
yf_data('global','nokusd','nokusd_rate','NOK=X', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >6.7 Russian Ruble vs USD</h5>

In [134]:
#yahoo finance ticker is RUB=X
yf_data('global','rubusd','rubusd_rate','RUB=X', '2016-01-01')

[*********************100%***********************]  1 of 1 completed


<h5 style="color: #CEDDF4;" >6.8 Swiss Franc vs USD</h5>

In [135]:
#yahoo finance ticker is CHF=X
yf_data('global','chfusd','chfusd_rate','CHF=X', '2016-01-01')

[*********************100%***********************]  1 of 1 completed
